# Setup — Shared Synthetic Signed Network

This notebook validates the shared **1,000-node / 7,500-directed-arc** signed network. The network is source data: do **not** regenerate or overwrite it unless the file is missing. Generation seed is 42 and the network is reused by every B1–B4 experiment.

In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
for p in [ROOT, ROOT / 'common', ROOT / 'cpp_engine']:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from network import NETWORK_CSV, NETWORK_METADATA_V2, load_network, validate_network, verify_fingerprint, write_metadata_v2

print('Project root:', ROOT)
print('Network file:', NETWORK_CSV)


In [ ]:
df, G, nodes = load_network(NETWORK_CSV)
report = validate_network(df)
print(f'nodes={len(nodes)}  directed_arcs={len(df)}  weak_components={report["n_weak_components"]}')
print(json.dumps(report, indent=2))


## Fingerprint and metadata

The v2 metadata stores the file SHA-256 and a semantic edge-table SHA-256. All scenario notebooks verify this fingerprint before running.

In [ ]:
if not NETWORK_METADATA_V2.exists():
    write_metadata_v2(df, NETWORK_CSV, NETWORK_METADATA_V2)
    print('Wrote', NETWORK_METADATA_V2)
check = verify_fingerprint(NETWORK_CSV, NETWORK_METADATA_V2)
print(json.dumps(check, indent=2))
assert check['ok'], 'Network fingerprint mismatch: stop and investigate.'


## Optional seed-42 regeneration check

This is off by default. It regenerates the network in memory and compares it to the stored file; it never overwrites the stored network.

In [ ]:
REGENERATE_CHECK = False
if REGENERATE_CHECK:
    import pandas as pd
    from network import regenerate_network_df
    regenerated = regenerate_network_df()
    pd.testing.assert_frame_equal(regenerated.reset_index(drop=True), df.reset_index(drop=True))
    print('Seed-42 regeneration matches the stored network.')
else:
    print('Regeneration check skipped (set REGENERATE_CHECK=True to verify).')


## C++ engine status

Build the extension once from a terminal in the project root (no installation is performed):

```bash
cd cpp_engine && python setup.py build_ext --inplace
```

In [ ]:
from cpp_adapter import cpp_available, openmp_available, cpp_import_error
print('C++ available:', cpp_available())
print('OpenMP:', openmp_available())
if not cpp_available():
    print('Import error:', cpp_import_error())
    print('Build manually: cd cpp_engine && python setup.py build_ext --inplace')


## Next steps

Run `B1_Random_opponent.ipynb`, `B2_Degree-based_opponent.ipynb`, `B3_Positive_PageRank_opponent.ipynb`, and `B4_Community-concentrated_opponent.ipynb` in any order, then `05_Analysis_Charts.ipynb`. For robust overnight execution, prefer the CLI:

```bash
python common/run_experiment.py --print-plan
python common/run_experiment.py --run-id run_YYYYMMDD_HHMMSS
```